# v6 Significance — paired bootstrap

`final_model_v6.ipynb` reported **macro-F1 0.6485 vs v5's 0.6363, a margin of +0.0122**. The marginal
noise band established in `ablation_significance.ipynb` is **+/-0.012**, so v6 clears it by 0.0002 -
far too fine to rest on. Two further reasons to be careful:

- **Worms accounts for 57% of the gain** (0.070 of the 0.122 summed per-class delta) on **48 test
  rows** with a +/-0.13 interval. Excluding Worms the gain is +0.0052, inside noise.
- The right instrument for comparing two configurations on the *same* test set is the **paired**
  interval (~+/-0.012 marginal, but roughly +/-0.012 -> tighter when paired), not the marginal one.
  `ablation_significance.ipynb` is explicit: *"TWO YARDSTICKS, don't mix them."*

This notebook applies the same protocol: **2,000 stratified bootstrap resamples of the test set**,
with every configuration scored on the **same resample** each iteration, so the differences are paired.

## Configurations compared

| Tag | Features | Combiner |
|---|---|---|
| `v3` | BASE 44 | static per-class $W$ |
| `v5` | BASE 44 | DEC dynamic, k=100 p=8 |
| **`v6`** | **DERIVED 75** | **DEC dynamic, k=30 p=8** |
| `single-BASE` | BASE 44 | none (alpha=0.25) |
| `single-DERIVED` | DERIVED 75 | none (alpha=0.25) |

The decomposition `single-BASE -> single-DERIVED` isolates the **feature** effect and
`v3 -> v5` the **combiner** effect, so the compounding claim is testable rather than asserted.

> **Runtime ~6 min.** Both OOF caches are reused; only the pool refits and the bootstrap loop run.

## 1. Rebuild both feature spaces and both pools

In [1]:
import numpy as np, pandas as pd, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign",1:"Analysis",2:"Backdoor",3:"DoS",4:"Exploits",
          5:"Fuzzers",6:"Generic",7:"Reconnaissance",8:"Shellcode",9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
RS, EPS = 42, 1e-6

Xraw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
yraw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([Xraw, yraw], axis=1).drop_duplicates().reset_index(drop=True)
Xraw, y = df[Xraw.columns], df["Label"]
nun = Xraw.nunique(); zero = nun[nun <= 1].index.tolist()
FULL = Xraw.drop(columns=zero)
corr = FULL.corr(); upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red = [c for c in upper.columns if (upper[c] >= 0.95).any()]
BASE = FULL.drop(columns=red)

def safe(a, b): return (a / (b + EPS)).replace([np.inf, -np.inf], 0.0).fillna(0.0)
F = FULL; g = lambda c: F[c] if c in F.columns else pd.Series(0.0, index=F.index)
fpk, bpk = g("Total Fwd Packet"), g("Total Bwd packets")
fby, bby = g("Total Length of Fwd Packet"), g("Total Length of Bwd Packet")
tot_pk, tot_by = fpk + bpk, fby + bby
d = {"r_pkt_fwd_bwd": safe(fpk, bpk), "r_byte_fwd_bwd": safe(fby, bby),
     "asym_pkt": safe(fpk - bpk, tot_pk), "asym_byte": safe(fby - bby, tot_by),
     "bpp_fwd": safe(fby, fpk), "bpp_bwd": safe(bby, bpk), "bpp_flow": safe(tot_by, tot_pk)}
d["r_bpp"] = safe(d["bpp_fwd"], d["bpp_bwd"])
d["cv_flow_iat"] = safe(g("Flow IAT Std"), g("Flow IAT Mean"))
d["cv_fwd_iat"] = safe(g("Fwd IAT Std"), g("Fwd IAT Mean"))
d["cv_bwd_iat"] = safe(g("Bwd IAT Std"), g("Bwd IAT Mean"))
d["iat_span_rel"] = safe(g("Flow IAT Max") - g("Flow IAT Min"), g("Flow IAT Mean"))
d["cv_fwd_len"] = safe(g("Fwd Packet Length Std"), g("Fwd Packet Length Mean"))
d["cv_bwd_len"] = safe(g("Bwd Packet Length Std"), g("Bwd Packet Length Mean"))
d["cv_pkt_len"] = safe(g("Packet Length Std"), g("Packet Length Mean"))
d["fwd_len_span_rel"] = safe(g("Fwd Packet Length Max") - g("Fwd Packet Length Min"), g("Fwd Packet Length Mean"))
d["bwd_len_span_rel"] = safe(g("Bwd Packet Length Max") - g("Bwd Packet Length Min"), g("Bwd Packet Length Mean"))
for fl in ("FIN", "SYN", "RST", "PSH", "ACK"): d["frac_" + fl] = safe(g(fl + " Flag Count"), tot_pk)
d["r_syn_fin"] = safe(g("SYN Flag Count"), g("FIN Flag Count"))
d["hdr_per_pkt_fwd"] = safe(g("Fwd Header Length"), fpk)
d["hdr_per_pkt_bwd"] = safe(g("Bwd Header Length"), bpk)
d["payload_share_fwd"] = safe(fby - g("Fwd Header Length"), fby)
dur = g("Flow Duration")
d["active_share"] = safe(g("Active Mean"), dur); d["idle_share"] = safe(g("Idle Mean"), dur)
d["r_active_idle"] = safe(g("Active Mean"), g("Idle Mean")); d["dur_per_pkt"] = safe(dur, tot_pk)
d["r_init_win"] = safe(g("FWD Init Win Bytes"), g("Bwd Init Win Bytes"))
DER = pd.DataFrame(d, index=F.index).replace([np.inf, -np.inf], 0.0).fillna(0.0)
dz = DER.nunique(); DER = DER.drop(columns=dz[dz <= 1].index.tolist())
DERIVED = pd.concat([BASE, DER], axis=1)

idx_all = np.arange(len(y))
itr, ite = train_test_split(idx_all, test_size=0.2, random_state=RS, stratify=y)
y_arr = y.values; att = y_arr != 0
labs = sorted(l for l in np.unique(y_arr) if l != 0)
f2i = {f: i for i, f in enumerate(labs)}; i2f = {i: f for f, i in f2i.items()}
NC = len(labs)
fam_all = np.full(len(y_arr), -1); fam_all[att] = pd.Series(y_arr[att]).map(f2i).values
y_test_arr = y_arr[ite]
print("BASE %d | DERIVED %d | test %d" % (BASE.shape[1], DERIVED.shape[1], len(ite)))

BASE 44 | DERIVED 75 | test 61235


In [2]:
def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS); p.update(kw); return XGBClassifier(**p)
def wc_alpha(n, a):
    w = (n.sum() / n) ** a; return w / w.mean()
def sample_w(ya, wc):
    w = wc[ya].astype(float); return w * len(w) / w.sum()
SPECS = ([ (f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0) ]
         + [("XGB smote2000", ("smote", 2000)), ("RF a=0.25", ("rf", 0.25)), ("ET a=0.25", ("et", 0.25))])
MODELS = [s[0] for s in SPECS]; M = len(MODELS)
def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if kind == "alpha": m = make_xgb()
    elif kind == "rf":  m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":  m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param))); return m
def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    gg = make_xgb(); gg.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb)); return gg

GRID_W = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)
def fuse_static(P, W):
    S = np.zeros(P[0].shape, np.float64)
    for m in range(M): S += P[m] * W[m][None, :]
    return S.argmax(1)
def fit_W(P, yt, passes=3):
    W = np.ones((M, NC)) / M
    best = f1_score(yt, fuse_static(P, W), average="macro", zero_division=0)
    for _ in range(passes):
        for c in range(NC):
            for m_ in range(M):
                cur = W[m_, c]
                for gg in GRID_W:
                    W[m_, c] = gg
                    if W[:, c].sum() == 0: continue
                    s = f1_score(yt, fuse_static(P, W), average="macro", zero_division=0)
                    if s > best + 1e-7: best, cur = s, gg
                W[m_, c] = cur
    return W
print("helpers ready")

helpers ready


## 2. Generate test predictions for every configuration

Each space is refit once on all of `train+val`; the cached OOF probabilities supply the combiner
parameters (per-class $W$ and the competence reference set) exactly as their own notebooks did.

In [3]:
comb_idx, selv_idx = train_test_split(np.arange(len(y_arr[itr])), test_size=0.5,
                                      random_state=RS + 1, stratify=y_arr[itr])
att_tr = att[itr]
ref_local = comb_idx[att_tr[comb_idx]]
fam_tr = fam_all[itr]
y_ref = fam_tr[ref_local]
WC_REF = wc_alpha(np.bincount(y_ref, minlength=NC), 0.25)
prof = lambda ps: np.hstack(ps).astype(np.float32)

PRED = {}
for tag, Xs, cache, kbest in (("BASE", BASE, "cache_oof_v4.npz", 100),
                              ("DERIVED", DERIVED, "cache_oof_v6_derived.npz", 30)):
    z = np.load(cache)
    OOF = [z[f"m{m}"] for m in range(M)]
    Xtr, Xte = Xs.iloc[itr].reset_index(drop=True), Xs.iloc[ite]
    t0 = time.time()
    gate = fit_gate(Xtr, y_arr[itr])
    Xa = Xtr[att_tr].reset_index(drop=True); ya = pd.Series(fam_tr[att_tr])
    POOL = [fit_member(k, p, Xa, ya) for _, (k, p) in SPECS]
    single = fit_member("alpha", 0.25, Xa, ya)

    gp = gate.predict(Xte); gm = gp == 1; gpos = np.where(gm)[0]
    P = [m.predict_proba(Xte.iloc[gpos]).astype(np.float32) for m in POOL]

    CORRECT = np.stack([(OOF[m][ref_local].argmax(1) == y_ref) for m in range(M)])
    NN_D = NearestNeighbors(n_neighbors=100, algorithm="brute", n_jobs=-1).fit(
        prof([OOF[m][ref_local] for m in range(M)]))
    _, ii = NN_D.kneighbors(prof(P))
    W_cls = fit_W([OOF[m][ref_local] for m in range(M)], y_ref)

    def dyn(k, p):
        nb = ii[:, :k]; wn = WC_REF[y_ref[nb]]; den = wn.sum(1)
        Wd = np.empty((nb.shape[0], M))
        for m in range(M): Wd[:, m] = (CORRECT[m][nb] * wn).sum(1) / den
        Wp = np.power(np.clip(Wd, 1e-6, None), p); Wp = Wp / Wp.sum(1, keepdims=True)
        S = np.zeros(P[0].shape, np.float64)
        for m in range(M): S += Wp[:, m:m + 1] * P[m]
        return S.argmax(1)

    def route(fam_g):
        yp = np.zeros(len(y_test_arr), int)
        yp[gm] = np.vectorize(i2f.get, otypes=[int])(fam_g); return yp

    if tag == "BASE":
        PRED["v3"] = route(fuse_static(P, W_cls))
        PRED["v5"] = route(dyn(100, 8))
        PRED["single-BASE"] = route(single.predict(Xte.iloc[gpos]))
    else:
        PRED["v6"] = route(dyn(30, 8))
        PRED["single-DERIVED"] = route(single.predict(Xte.iloc[gpos]))
    print("  %-8s done (%.0fs)" % (tag, time.time() - t0))

print()
for k in ("v3", "v5", "v6", "single-BASE", "single-DERIVED"):
    print("  %-16s macro-F1 %.4f  acc %.4f"
          % (k, f1_score(y_test_arr, PRED[k], average="macro", zero_division=0),
             (PRED[k] == y_test_arr).mean()))

  BASE     done (80s)


  DERIVED  done (111s)

  v3               macro-F1 0.6334  acc 0.9086
  v5               macro-F1 0.6363  acc 0.9089
  v6               macro-F1 0.6485  acc 0.9123
  single-BASE      macro-F1 0.6157  acc 0.9081
  single-DERIVED   macro-F1 0.6258  acc 0.9117


## 3. Paired stratified bootstrap — 2,000 resamples

Every configuration is scored on the **same** resample each iteration, so differences are paired and
the interval reflects only the disagreement between configurations, not the marginal variability of
either.

In [4]:
B = 2000
rng = np.random.default_rng(RS)
strata = [np.where(y_test_arr == c)[0] for c in range(10)]
KEYS = ["v3", "v5", "v6", "single-BASE", "single-DERIVED"]
no_worms = [c for c in range(10) if c != 9]

boot = {k: np.empty(B) for k in KEYS}
boot_nw = {k: np.empty(B) for k in KEYS}
boot_acc = {k: np.empty(B) for k in KEYS}
t0 = time.time()
for b in range(B):
    idx = np.concatenate([rng.choice(s, len(s), replace=True) for s in strata])
    yt = y_test_arr[idx]
    for k in KEYS:
        yp = PRED[k][idx]
        boot[k][b] = f1_score(yt, yp, average="macro", zero_division=0)
        boot_nw[k][b] = f1_score(yt, yp, average="macro", labels=no_worms, zero_division=0)
        boot_acc[k][b] = (yp == yt).mean()
print("%d resamples in %.0fs" % (B, time.time() - t0))

def ci(a): return np.percentile(a, [2.5, 97.5])
print("\n--- marginal intervals (for REPORTING a single config) ---")
for k in KEYS:
    lo, hi = ci(boot[k])
    print("  %-16s %.4f  [%.4f, %.4f]  width %.4f" % (k, boot[k].mean(), lo, hi, hi - lo))

print("\n--- PAIRED differences (for COMPARING two configs) ---")
pairs = [("v6", "v5", "features, given the dynamic combiner"),
         ("v6", "v3", "full: features + combiner vs old best"),
         ("single-DERIVED", "single-BASE", "feature effect, no combiner"),
         ("v5", "v3", "combiner effect, BASE features"),
         ("v6", "single-DERIVED", "combiner effect, DERIVED features")]
rows = []
for a, b_, why in pairs:
    d = boot[a] - boot[b_]
    lo, hi = ci(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    rows.append(dict(comparison=f"{a} - {b_}", delta=round(d.mean(), 4),
                     lo=round(lo, 4), hi=round(hi, 4), p=round(p, 4),
                     established="YES" if lo > 0 or hi < 0 else "no", what=why))
print(pd.DataFrame(rows).to_string(index=False))
print("\nmean paired CI half-width: %.4f" % np.mean([(r["hi"] - r["lo"]) / 2 for r in rows]))

2000 resamples in 72s

--- marginal intervals (for REPORTING a single config) ---
  v3               0.6333  [0.6150, 0.6512]  width 0.0363
  v5               0.6362  [0.6173, 0.6539]  width 0.0366
  v6               0.6485  [0.6302, 0.6669]  width 0.0367
  single-BASE      0.6154  [0.5966, 0.6346]  width 0.0380
  single-DERIVED   0.6255  [0.6050, 0.6457]  width 0.0406

--- PAIRED differences (for COMPARING two configs) ---
                  comparison  delta      lo     hi     p established                                  what
                     v6 - v5 0.0123  0.0012 0.0233 0.032         YES  features, given the dynamic combiner
                     v6 - v3 0.0152  0.0039 0.0272 0.012         YES full: features + combiner vs old best
single-DERIVED - single-BASE 0.0100 -0.0008 0.0211 0.071          no           feature effect, no combiner
                     v5 - v3 0.0029 -0.0060 0.0127 0.555          no        combiner effect, BASE features
         v6 - single-DERIVED 0.0231  

### 3.1 Robustness — does v6 survive without Worms?

Worms contributes 57% of the raw v6-v5 delta on 48 test rows. Macro-F1 over the other nine classes is
the check that matters.

In [5]:
print("--- macro-F1 EXCLUDING Worms ---")
for k in KEYS:
    lo, hi = ci(boot_nw[k])
    print("  %-16s %.4f  [%.4f, %.4f]" % (k, boot_nw[k].mean(), lo, hi))
print()
rows = []
for a, b_ in (("v6", "v5"), ("v6", "v3"), ("single-DERIVED", "single-BASE")):
    d = boot_nw[a] - boot_nw[b_]; lo, hi = ci(d)
    p = 2 * min((d <= 0).mean(), (d >= 0).mean())
    rows.append(dict(comparison=f"{a} - {b_} (no Worms)", delta=round(d.mean(), 4),
                     lo=round(lo, 4), hi=round(hi, 4), p=round(p, 4),
                     established="YES" if lo > 0 or hi < 0 else "no"))
print(pd.DataFrame(rows).to_string(index=False))

print("\n--- accuracy, paired ---")
for a, b_ in (("v6", "v5"), ("v6", "v3")):
    d = boot_acc[a] - boot_acc[b_]; lo, hi = ci(d)
    print("  %-12s %+.4f  [%+.4f, %+.4f]  %s"
          % (f"{a} - {b_}", d.mean(), lo, hi, "YES" if lo > 0 or hi < 0 else "no"))

--- macro-F1 EXCLUDING Worms ---
  v3               0.6511  [0.6373, 0.6639]
  v5               0.6501  [0.6357, 0.6640]
  v6               0.6557  [0.6401, 0.6703]
  single-BASE      0.6378  [0.6224, 0.6532]
  single-DERIVED   0.6437  [0.6279, 0.6593]

                             comparison  delta      lo     hi     p established
                     v6 - v5 (no Worms) 0.0056 -0.0035 0.0142 0.211          no
                     v6 - v3 (no Worms) 0.0046 -0.0048 0.0137 0.325          no
single-DERIVED - single-BASE (no Worms) 0.0059 -0.0041 0.0157 0.234          no

--- accuracy, paired ---
  v6 - v5      +0.0034  [+0.0025, +0.0044]  YES
  v6 - v3      +0.0037  [+0.0026, +0.0047]  YES


### 3.2 Per-class intervals for v6

In [6]:
rows = []
for c in range(10):
    vals = np.empty(B)
    for b in range(B):
        idx = np.concatenate([rng.choice(s, len(s), replace=True) for s in strata])
        vals[b] = f1_score(y_test_arr[idx], PRED["v6"][idx], average=None,
                           labels=[c], zero_division=0)[0]
    lo, hi = ci(vals)
    rows.append(dict(family=NAMES[c], support=int((y_test_arr == c).sum()),
                     f1=round(f1_score(y_test_arr, PRED["v6"], average=None, labels=[c],
                                       zero_division=0)[0], 3),
                     lo=round(lo, 3), hi=round(hi, 3), width=round(hi - lo, 3)))
print(pd.DataFrame(rows).sort_values("support", ascending=False).to_string(index=False))

        family  support    f1    lo    hi  width
        Benign    45001 0.985 0.984 0.986  0.002
      Exploits     6182 0.791 0.783 0.798  0.015
       Fuzzers     5279 0.721 0.714 0.728  0.014
Reconnaissance     2406 0.640 0.623 0.657  0.034
       Generic      899 0.828 0.809 0.847  0.038
           DoS      883 0.468 0.436 0.501  0.065
     Shellcode      370 0.388 0.344 0.434  0.090
      Backdoor       90 0.635 0.541 0.720  0.179
      Analysis       77 0.446 0.378 0.516  0.138
         Worms       48 0.582 0.447 0.699  0.251


## 4. Findings

**v6 is established over both v5 and v3 on full macro-F1 and on accuracy. The macro-F1 gain does
depend materially on Worms; the accuracy gain does not.**

2,000 stratified paired resamples. Mean paired CI half-width **0.0112**, consistent with the +/-0.012
band from `ablation_significance.ipynb`.

### 1. Headline comparisons (paired, full 10-class macro-F1)

| Comparison | delta | 95% CI | p | established |
|---|---|---|---|---|
| **v6 - v5** | **+0.0123** | [0.0012, 0.0233] | **0.032** | **YES** |
| **v6 - v3** | **+0.0152** | [0.0039, 0.0272] | **0.012** | **YES** |
| **v6 - single-DERIVED** | **+0.0231** | [0.0104, 0.0361] | **<0.001** | **YES** |
| single-DERIVED - single-BASE | +0.0100 | [-0.0008, 0.0211] | 0.071 | no (marginal) |
| v5 - v3 | +0.0029 | [-0.0060, 0.0127] | 0.555 | no |

### 2. Accuracy — the most robust result in the notebook

| Comparison | delta | 95% CI | established |
|---|---|---|---|
| **v6 - v5** | **+0.0034** | [+0.0025, +0.0044] | **YES** |
| **v6 - v3** | **+0.0037** | [+0.0026, +0.0047] | **YES** |

These intervals are an order of magnitude tighter than the macro-F1 ones and nowhere near zero.
Accuracy is dominated by the large classes, so it is **independent of the Worms concern entirely**.
**v6 = accuracy 0.9123 is the least equivocal claim in this project.**

### 3. ⚠️ The macro-F1 gain does not survive removing Worms

| Comparison (macro-F1 over the 9 non-Worms families) | delta | 95% CI | p | established |
|---|---|---|---|---|
| v6 - v5 (no Worms) | +0.0056 | [-0.0035, 0.0142] | 0.211 | **no** |
| v6 - v3 (no Worms) | +0.0046 | [-0.0048, 0.0137] | 0.325 | **no** |
| single-DERIVED - single-BASE (no Worms) | +0.0059 | [-0.0041, 0.0157] | 0.234 | **no** |

**State this whenever the macro-F1 number is quoted.** Worms has 48 test rows and a per-class interval
of **[0.447, 0.699]** (width 0.251). The full-macro result is real - macro-F1 weights all ten classes
equally *by design*, and a genuine improvement on the rarest class is a legitimate improvement - but a
reader is entitled to know that removing one 48-row class makes it disappear.

The defensible pair of sentences:

> v6 improves 10-class macro-F1 by +0.0123 over v5 (95% CI [0.0012, 0.0233], p=0.032) and accuracy by
> +0.0034 ([0.0025, 0.0044]). The macro-F1 gain is concentrated in the rarest family (Worms, 48 test
> rows); restricted to the other nine families the difference is +0.0056 and not statistically
> established.

### 4. CORRECTION: this is an interaction, not a feature effect

`final_model_v6.ipynb` concluded *"attribute the gain to the features, not the dynamic component."*
**The bootstrap does not support that attribution.** Neither ingredient is established alone:

| Ingredient, in isolation | delta | p |
|---|---|---|
| features, no ensemble (`single-DERIVED - single-BASE`) | +0.0100 | 0.071 |
| dynamic vs static combiner, BASE features (`v5 - v3`) | +0.0029 | 0.555 |
| **both together (`v6 - v3`)** | **+0.0152** | **0.012** |

Yet **ensembling on DERIVED features is strongly established** (`v6 - single-DERIVED` = +0.0231,
p<0.001) while ensembling on BASE was not decisive. **The combiner only pays once the features are
good enough** - which is exactly the mechanism v5 identified: competence estimation needs pure
neighbourhoods, and derived features are what made them pure (purity 0.357 -> 0.376, and the optimal
k fell 100 -> 30).

This is the **same structural pattern as the project's earlier headline finding**, where the hierarchy
was significant only in combination with alpha=0.25 weighting (+0.0178) and not on its own (-0.0061,
n.s.). **Frame v6 as "derived features + ensemble jointly" - an interaction, not a main effect** - and
do not claim either ingredient carries it alone.

Note this does *not* rehabilitate dynamic-over-static: on DERIVED, `DYN k=30 p=8` (0.6485) versus
`STATIC per-class W` (0.6459) is +0.0026, inside noise. **The established effect is ensembling, not
dynamism.**

### 5. Per-class intervals for v6 - report rare families as intervals

| Family | support | F1 | 95% CI | width |
|---|---|---|---|---|
| Benign | 45,001 | 0.985 | [0.984, 0.986] | 0.002 |
| Exploits | 6,182 | 0.791 | [0.783, 0.798] | 0.015 |
| Fuzzers | 5,279 | 0.721 | [0.714, 0.728] | 0.014 |
| Reconnaissance | 2,406 | 0.640 | [0.623, 0.657] | 0.034 |
| Generic | 899 | 0.828 | [0.809, 0.847] | 0.038 |
| DoS | 883 | 0.468 | [0.436, 0.501] | 0.065 |
| Shellcode | 370 | 0.388 | [0.344, 0.434] | 0.090 |
| Backdoor | 90 | 0.635 | [0.541, 0.720] | 0.179 |
| Analysis | 77 | 0.446 | [0.378, 0.516] | 0.138 |
| **Worms** | **48** | **0.582** | **[0.447, 0.699]** | **0.251** |

### Recommendation

**Report v6 as the final model**: `macro-F1 0.649 [0.630, 0.667]`, `accuracy 0.9123`.

Three sentences that are all defensible:

1. v6 improves macro-F1 over the previous best by **+0.0123 (p=0.032)** and accuracy by **+0.0034
   (p<0.001)**.
2. The macro-F1 gain is **concentrated in Worms**; over the other nine families it is +0.0056 and not
   established.
3. The improvement is an **interaction between derived features and ensembling** - neither is
   established in isolation (p=0.071 and p=0.555 respectively).

Intervals capture test-set sampling only; split and seed variance are excluded, so they remain
optimistic. `NEXT_STEPS.md` 3.2 (multi-seed repeated splits) is the cheapest way to close that gap and
is now the most valuable non-feature work outstanding.